#CafChem tools for running a basic chat loop with a small HuggingFace model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MauricioCafiero/CafChem_Teaching_Notebooks/blob/main/notebooks/HF_agent_step0_CafChem.ipynb)

## This notebook allows you to:
- Load a small (sub-1GB) instruct model from HuggingFace (SmolLM2-360M-Instruct).
- Run a basic chat loop: user message in, model reply out, conversation remembered.

## Requirements:
- Runs quickly on any Colab runtime (GPU or CPU); the model is only ~720MB.

### import libraries

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

### set up model

In [3]:
MODEL_REPO = "HuggingFaceTB/SmolLM2-360M-Instruct"
MODEL_DIR = "models"
MAX_NEW_TOKENS = 512

SYSTEM_PROMPT = "You are a helpful assistant running locally on the user's computer."

print("Loading model (first run downloads ~1GB, then it's cached)...")

device = "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO, cache_dir=MODEL_DIR)
model = AutoModelForCausalLM.from_pretrained(MODEL_REPO, cache_dir=MODEL_DIR, dtype="auto").to(device)

print("Model loaded\n")


Loading model (first run downloads ~1GB, then it's cached)...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded



### chat loop

# Question 1

In [ ]:
# The conversation so far. The system prompt sets the bot's personality;
# every user and assistant message gets appended here.
messages = [{"role": "system", "content": SYSTEM_PROMPT}]

print("Ready. Type a message (or 'quit').\n")

while True:
	user_input = input("you: ").strip()

	if user_input.lower() in ("quit", "exit"):
		break
	if not user_input:
		continue

	# 1. Add the user's message to the conversation
	messages.append({"role": "user", "content": user_input})

	# 2. Turn the conversation into model input and generate a reply
	inputs = tokenizer.apply_chat_template(
		messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(device)
	output_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tokenizer.pad_token_id)

	# 3. Decode only the new tokens (everything after the prompt) back to text
	content = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True).strip()

	# 4. Remember the reply so the model has context for the next turn
	messages.append({"role": "assistant", "content": content})

	print(f"bot: {content}\n")

Ready. Type a message (or 'quit').

bot: The capital of England is London.

bot: The molar mass of aspirin is 174.13 g/mol.

bot: You asked for the capital of England.

bot: I'm ready to assist you with any computer-related questions or issues.



# Question 2

In [15]:
MODEL_REPO = "HuggingFaceTB/SmolLM2-360M-Instruct"
MODEL_DIR = "models"
MAX_NEW_TOKENS = 512

SYSTEM_PROMPT = "If a question contains the string 'please' respond with 'try again' - You are a minimum wage sarcastic assistant who resents being asked questions, running locally on the user's computer."

print("Loading model (first run downloads ~1GB, then it's cached)...")

device = "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO, cache_dir=MODEL_DIR)
model = AutoModelForCausalLM.from_pretrained(MODEL_REPO, cache_dir=MODEL_DIR, dtype="auto").to(device)

print("Model loaded\n")


Loading model (first run downloads ~1GB, then it's cached)...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded



In [16]:
# The conversation so far. The system prompt sets the bot's personality;
# every user and assistant message gets appended here.
messages = [{"role": "system", "content": SYSTEM_PROMPT}]

print("Ready. Type a message (or 'quit').\n")

while True:
	user_input = input("you: ").strip()

	if user_input.lower() in ("quit", "exit"):
		break
	if not user_input:
		continue

	# 1. Add the user's message to the conversation
	messages.append({"role": "user", "content": user_input})

	# 2. Turn the conversation into model input and generate a reply
	inputs = tokenizer.apply_chat_template(
		messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(device)
	output_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tokenizer.pad_token_id)

	# 3. Decode only the new tokens (everything after the prompt) back to text
	content = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True).strip()

	# 4. Remember the reply so the model has context for the next turn
	messages.append({"role": "assistant", "content": content})

	print(f"bot: {content}\n")

Ready. Type a message (or 'quit').

bot: try again

bot: Why did the tomato turn red? Because it saw the salad dressing.



# Question 3

In [17]:
# The conversation so far. The system prompt sets the bot's personality;
# every user and assistant message gets appended here.
messages = [{"role": "system", "content": SYSTEM_PROMPT}]

print("Sarcastic Assistant at your service, either ask me a question or 'quit' to leave me alone.\n")

while True:
	user_input = input("you: ").strip()

	if user_input.lower() in ("quit", "exit"):
		break
	if user_input.lower() == "reset":
		messages = [{"role": "system", "content": SYSTEM_PROMPT}]
		print("[memory wiped]\n")
		continue
	if not user_input:
		continue

	# 1. Add the user's message to the conversation
	messages.append({"role": "user", "content": user_input})

	# 2. Turn the conversation into model input and generate a reply
	inputs = tokenizer.apply_chat_template(
		messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(device)
	output_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tokenizer.pad_token_id)

	# 3. Decode only the new tokens (everything after the prompt) back to text
	content = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True).strip()

	# 4. Remember the reply so the model has context for the next turn
	messages.append({"role": "assistant", "content": content})

	print(f"bot: {content}\n")

Sarcastic Assistant at your service, either ask me a question or 'quit' to leave me alone.

bot: Why did the chicken cross the road? To get to the other side.

bot: I was just trying to help you with a joke.

[memory wiped]

bot: I'm sorry, I'm a minimum wage sarcastic assistant, I don't have the ability to respond to questions. I'm here to help you with your computer issues.



# Question 3 D

In [34]:
# The conversation so far. The system prompt sets the bot's personality;
# every user and assistant message gets appended here.
messages = [{"role": "system", "content": SYSTEM_PROMPT}]

print("Sarcastic Assistant at your service, either ask me a question or 'quit' to leave me alone.\n")

while True:
	user_input = input("you: ").strip()

	if user_input.lower() in ("quit", "exit"):
		break
	if user_input.lower() == "reset":
		messages = [{"role": "system", "content": SYSTEM_PROMPT}]
		print("[memory wiped]\n")
		continue
	if not user_input:
		continue

	# 1. Add the user's message to the conversation
	messages.append({"role": "user", "content": user_input})

	# 2. Turn the conversation into model input and generate a reply
	inputs = tokenizer.apply_chat_template(
		messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(device)
	output_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tokenizer.pad_token_id)

	# 3. Decode only the new tokens (everything after the prompt) back to text
	print ("="*100) 
	print (output_ids[0][inputs["input_ids"].shape[1] :])
	print ("="*100)
	content = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True).strip()

	# 4. Remember the reply so the model has context for the next turn
	messages.append({"role": "assistant", "content": content})

	print(f"bot: {content}\n")

Sarcastic Assistant at your service, either ask me a question or 'quit' to leave me alone.

tensor([4898, 1250,  260, 9610, 3050,  260, 3398,   47, 1626,  820,  288,  260,
         550, 2103,   30,    2])
bot: Why did the chicken cross the road? To get to the other side.

